# Compare BERTopic, LDA, and independent human review

This notebook implements the comparison in [PR2's Experimental Design and planned Results](../../progress-report-2/pr2-submission.tex): shared $c_v$ coherence, keyword diversity, computational cost,
top-ten-term Jaccard overlap, publication proportions, and two-reviewer article–topic fit.
Human review evaluates each model; it is not a third topic model or ground-truth labeling task.

Run modern notebooks 01–03, document the team selection, and obtain Laura's four LDA exports.
Then run this notebook top to bottom. It reads exported results without loading model pickles or
embeddings or refitting either pipeline. Missing selection, handoffs, labels, and ratings produce
explicit pending statuses. Invalid supplied artifacts raise errors. No historical PR2 numbers,
model choice, labels, or ratings are inserted as current results.

1. Validate the common corpus and model handoffs.
2. Compare quantitative metrics, keywords, assignments, publication patterns, and measured runtimes.
3. Prepare frozen packets after labels are confirmed; use the separate reviewer notebooks independently.
4. After both reviewers finish, enable the summary and export the combined comparison table.
5. Complete the interpretation prompts and use the evidence checklist for manuscript updates.

For continuity, artifacts remain under `modern-methods/Results/comparison/<modern-run>/` and
LDA handoffs remain under `modern-methods/Results/classical/`. Moving this notebook does not move
or replace existing answer files. Run-specific manifests bind outputs to both model exports.

In [1]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
ROOT = REPO/'final-project/code/modern-methods'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from shared.corpus import load_corpus, corpus_hash, digest, write_json
from shared import evaluation as ev
from src import artifacts as a
from src.reviewer import load_session
OUT = ROOT/'Results/modern'
CLASSICAL = ROOT/'Results/classical'  # Laura's existing export destination.
COMPARISON_ROOT = ROOT/'Results/comparison'
COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)

In [2]:
df = load_corpus()
reservation = json.loads((OUT/'review_sample.json').read_text())
if reservation['corpus_hash'] != corpus_hash(df):
    raise ValueError('Review reservation belongs to a different corpus')
if len(reservation['article_ids']) != len(set(reservation['article_ids'])) or not set(reservation['article_ids']).issubset(df.article_id):
    raise ValueError('Invalid reserved review IDs')
experiments = pd.read_csv(OUT/'experiments.csv')
selection = a.selection_record(OUT, df, experiments.run_id.tolist())
print(f'Common corpus: {len(df):,} articles; {df.publication.nunique()} publications.')
print(f'Reserved review sample: {len(reservation["article_ids"])} articles, independent of development review.')

Common corpus: 7,332 articles; 25 publications.
Reserved review sample: 100 articles, independent of development review.


## 1. Load the selected modern exports
The team selects a run in notebook 02; this notebook never chooses the highest-scoring run
implicitly. Validate the corpus, reservation, and exported file checksums before using results.
Unconfirmed keyword labels may appear in quantitative figures, but do not support named thematic claims.

In [3]:
modern = terms = inventory = labels = modern_manifest = None
labels_ready = False
if selection is None:
    print('Modern selection pending team article review; no run is chosen automatically.')
else:
    run_id = selection['run_id']
    folder = OUT/'runs'/run_id
    modern_manifest = json.loads((folder/'manifest.json').read_text())
    identity = modern_manifest['identity']
    if identity['corpus_hash'] != corpus_hash(df) or identity['reservation_hash'] != digest(reservation):
        raise ValueError('Modern run corpus or review reservation differs')
    for name in ['assignments.csv','terms.csv','topics.csv','representative_ids.csv','coherence.csv']:
        if hashlib.sha256((folder/name).read_bytes()).hexdigest() != modern_manifest['files'][name]:
            raise ValueError(f'Modern export changed: {name}')
    modern = pd.read_csv(folder/'assignments.csv',dtype={'article_id':str,'run_id':str})
    modern = ev.aligned_assignments(df, modern)
    if not modern.model.eq('bertopic').all() or not modern.run_id.eq(run_id).all():
        raise ValueError('Modern assignments do not describe the selected run')
    terms = pd.read_csv(folder/'terms.csv')
    selected_folder = OUT/'selected'/run_id
    labels = a.labels_table(folder, selected_folder, dict(run_id=run_id,corpus_hash=corpus_hash(df),run_identity_hash=digest(identity)))
    labels_ready = bool(len(labels) and labels.confirmed.astype(str).str.lower().eq('true').all())
    inventory = a.topic_inventory(folder, selected_folder, len(df), labels)
    display(inventory[['topic_id','label','label_status','count','prevalence']])
    print('Modern labels:', 'confirmed' if labels_ready else 'pending team confirmation')

Modern selection pending team article review; no run is chosen automatically.


## Laura's classical handoff
Place `assignments.csv`, `terms.csv`, `topics.csv`, and `manifest.json` in `Results/classical`.
The manifest must match the corpus hash. Assignments cover all article IDs exactly once;
terms include ten ranked unigrams per topic. Preserve LDA mixtures separately.

For independent human review, the manifest must additionally record `labels_confirmed: true`
and `review_reservation_hash` equal to the shared reservation's digest. This is Laura's
attestation that labels are frozen and the reserved articles were not development evidence.
Missing files are pending; malformed or misaligned supplied files are errors.

The manifest may include `timings` (stage → measured seconds) and `environment` (hardware/software). Missing timings remain unavailable; do not infer them from PR2.

In [4]:
required = ['assignments.csv','terms.csv','topics.csv','manifest.json']
missing = [name for name in required if not (CLASSICAL/name).exists()]
classical = classical_terms = classical_topics = classical_manifest = None
classical_labels_ready = False
COMPARISON = None
if missing:
    print('LDA handoff pending:', ', '.join(missing))
else:
    classical, classical_terms, classical_topics = ev.validate_classical(CLASSICAL, df.article_id.tolist())
    classical_manifest = json.loads((CLASSICAL/'manifest.json').read_text())
    if classical_manifest.get('corpus_hash') != corpus_hash(df):
        raise ValueError('LDA corpus hash differs')
    if classical_manifest.get('review_reservation_hash') not in [None, digest(reservation)]:
        raise ValueError('Classical review reservation differs from the frozen shared sample')
    classical_labels_ready = (classical_manifest.get('labels_confirmed') is True and
                              classical_manifest.get('review_reservation_hash') == digest(reservation))
    if classical_labels_ready and classical_topics.label.fillna('').str.strip().eq('').any():
        raise ValueError('Confirmed classical labels cannot be blank')

    timings = classical_manifest.get('timings', {})
    if not isinstance(timings, dict) or any(isinstance(v, bool) or not isinstance(v, (int, float)) or not np.isfinite(v) or v < 0 for v in timings.values()):
        raise ValueError('LDA timings must map stage names to finite nonnegative seconds')
    top = classical_terms.query('rank <= 10')
    if top.duplicated(['topic_id','term']).any() or top.term.astype(str).str.contains(r'\s').any():
        raise ValueError('LDA top-ten terms must be distinct unigrams within each topic')

LDA handoff pending: assignments.csv, terms.csv, topics.csv, manifest.json


In [5]:
# Guard output identity before writing tables or touching a review packet.
COMPARISON = None
if modern is not None and classical is not None:
    COMPARISON = COMPARISON_ROOT/selection['run_id']
    comparison_identity = dict(
        corpus_hash=corpus_hash(df), modern_run=selection['run_id'],
        classical_run=str(classical.run_id.iloc[0]), classical_manifest_hash=digest(classical_manifest),
        modern_identity_hash=digest(modern_manifest['identity']),
        modern_files=modern_manifest['files'],
        classical_files={name:hashlib.sha256((CLASSICAL/name).read_bytes()).hexdigest() for name in required},
        reservation_hash=digest(reservation))
    binding = COMPARISON/'comparison_identity.json'
    if binding.exists():
        previous = json.loads(binding.read_text())
        if any(comparison_identity.get(key) != value for key, value in previous.items()):
            raise ValueError('Comparison belongs to different model exports; preserve old results before changing runs')
    COMPARISON.mkdir(parents=True, exist_ok=True)
    write_json(binding, comparison_identity)
else:
    print('Joint comparison pending selected modern model and validated LDA exports.')

Joint comparison pending selected modern model and validated LDA exports.


## 2. Shared quantitative evaluation
Both models use the same article reference, tokenizer, top-ten unigrams, and $c_v$ window of
110 tokens. Missing reference words are dropped for scoring; report vocabulary coverage and
per-topic values with the mean. Exclude BERTopic −1 from coherence and keyword diversity.
Diversity is unique top-ten terms divided by all top-ten slots. These are descriptive metrics,
not accuracy against true topic labels; more topics need not mean better themes.

In [6]:
comparison_metrics = None
if COMPARISON is not None:
    texts, dictionary = ev.reference_texts(df)
    metric_rows = []
    for model_name, model_terms, assignments in [('bertopic',terms,modern),('lda',classical_terms,classical)]:
        scores = ev.coherence(model_terms, texts, dictionary)
        scores.to_csv(COMPARISON/f'{"classical" if model_name == "lda" else "modern"}_coherence.csv',index=False)
        top = model_terms.query('topic_id >= 0 and rank <= 10')
        metric_rows.append(dict(model=model_name, topic_count=top.topic_id.nunique(),
            coherence=scores.coherence.mean(), vocabulary_coverage=scores.vocabulary_coverage.mean(),
            keyword_diversity=top.term.nunique()/len(top), outlier_fraction=assignments.topic_id.eq(-1).mean()))
    comparison_metrics = pd.DataFrame(metric_rows)
    comparison_metrics.to_csv(COMPARISON/'model_metrics.csv',index=False)
    display(comparison_metrics)
else:
    print('Shared quantitative metrics pending both model exports.')

Shared quantitative metrics pending both model exports.


## 3. Topic overlap and assignment contingency
**Figure caption:** Jaccard overlap of top-ten unigram sets; LDA topics are rows and BERTopic
topics are columns. Exclude −1. Report the best match for every LDA topic, including ties.
Matches can be many-to-one; zero overlap is not evidence of different meanings, and similar
words are not proof of semantic equivalence. Topic numbers have no cross-model meaning.

The separate assignment contingency table uses one dominant LDA topic per article and one
BERTopic assignment (including −1). It measures shared membership, not classification accuracy.
Read development representatives alongside labels before interpreting a match.

In [7]:
best = None
if COMPARISON is not None:
    matrix = ev.jaccard_table(classical_terms, terms)
    matrix.index.name = 'lda_topic'
    matrix.columns.name = 'modern_topic'
    matrix.to_csv(COMPARISON/'jaccard.csv')
    best = pd.DataFrame({'lda_topic':matrix.index,'modern_topic':matrix.idxmax(axis=1).values,'jaccard':matrix.max(axis=1).values})
    best['tied_modern_topics'] = [' | '.join(map(str,row.index[row.eq(row.max())])) for _,row in matrix.iterrows()]
    best['interpretation'] = np.where(best.jaccard.eq(0),'No lexical overlap; numeric tie is arbitrary','Best lexical match, not semantic equivalence')
    best.to_csv(COMPARISON/'best_lexical_matches.csv',index=False)
    display(best)
    fig, ax = plt.subplots(figsize=(12,8))
    image = ax.imshow(matrix,aspect='auto',vmin=0,vmax=1)
    ax.set_yticks(range(len(matrix)),matrix.index)
    ax.set_xticks(range(len(matrix.columns)),matrix.columns,rotation=90)
    ax.set(xlabel='BERTopic topic',ylabel='LDA topic',title='Top-ten unigram Jaccard overlap')
    fig.colorbar(image,ax=ax,label='Jaccard similarity')
    fig.tight_layout()
    for extension in ['png','pdf']: fig.savefig(COMPARISON/f'jaccard.{extension}',dpi=180)
    plt.show()
    plt.close(fig)
    paired = modern[['article_id','topic_id']].merge(classical[['article_id','topic_id']],on='article_id',validate='one_to_one',suffixes=('_modern','_lda'))
    contingency = pd.crosstab(paired.topic_id_lda,paired.topic_id_modern)
    contingency.to_csv(COMPARISON/'assignment_contingency.csv')
    display(contingency)
else:
    print('Topic overlap pending both model exports.')

Topic overlap pending both model exports.


## 4. Publication patterns in both models
**Figure captions:** Each heatmap shows within-publication shares of all articles, publication
sample sizes, outlets with at least 50 articles, and the 15 largest topics plus unassigned.
LDA has no unassigned articles (the compatibility column is zero). Full CSV matrices include
all 25 publications and every assigned topic. The displayed subset need not sum to 100%.

The matched-share table joins every publication to each LDA topic's best lexical modern match.
It includes ties, zero-overlap warnings, and both model-specific shares. Do not subtract these
as estimates of the same latent theme: matches can be weak or many-to-one, and models differ
in granularity. Compare broad patterns with article evidence and unequal date coverage in mind.

In [8]:
if COMPARISON is not None:
    publication_tables = {}
    for name, assignments, topic_labels in [
        ('modern',modern,dict(zip(inventory.topic_id,inventory.label))),
        ('classical',classical,dict(zip(classical_topics.topic_id,classical_topics.label)))]:
        counts, proportions = ev.publication_exports(df, assignments, COMPARISON/name, topic_labels)
        publication_tables[name] = (counts, proportions)
        plot = COMPARISON/name/'publication_proportions.png'
        if plot.exists(): display(Image(filename=str(plot)))
    rows = []
    for match in best.itertuples():
        for publication in publication_tables['classical'][1].index:
            lda_share = publication_tables['classical'][1].loc[publication].get(match.lda_topic, 0.0)
            modern_share = publication_tables['modern'][1].loc[publication].get(match.modern_topic, 0.0)
            rows.append(dict(publication=publication,
                article_count=int(publication_tables['classical'][0].loc[publication].sum()),
                lda_topic=match.lda_topic, modern_topic=match.modern_topic, jaccard=match.jaccard,
                tied_modern_topics=match.tied_modern_topics, interpretation=match.interpretation,
                lda_proportion=lda_share, modern_proportion=modern_share))
    matched_shares = pd.DataFrame(rows)
    matched_shares.to_csv(COMPARISON/'matched_publication_proportions.csv',index=False)
    display(matched_shares.head(10))
else:
    print('Cross-model publication comparison pending both model exports.')

Cross-model publication comparison pending both model exports.


## 5. Computational cost and robustness context
Report measured stages with hardware and software context. BERTopic fitting includes reduction,
clustering, and representation; evaluation is timed separately. Embedding preparation is a shared
up-front cost across six fits, not a cost to add six times. Cache reuse and missing LDA timings
prevent a simple fair total-runtime ranking. No stage-specific times are reconstructed.

Read each pipeline's seed-stability analysis separately: modern ARI excludes articles unassigned
in either run and must be accompanied by the compared fraction and outlier agreement. Do not
use historical PR2 stability values as current results or compare topic IDs across seeds.

In [9]:
runtime_table = None
if COMPARISON is not None:
    modern_metrics = modern_manifest['metrics']
    runtime = [dict(model='bertopic',stage='fitting and representation',seconds=modern_metrics['fitting_representation_seconds']),
               dict(model='bertopic',stage='evaluation',seconds=modern_metrics['evaluation_seconds'])]
    for stage, seconds in classical_manifest.get('timings',{}).items():
        runtime.append(dict(model='lda',stage=stage,seconds=seconds))
    embedding_context = None
    execution_path = OUT/'embedding_execution.json'
    if execution_path.exists():
        pointer = json.loads((OUT/'embedding_pointer.json').read_text())
        if pointer['array_hash'] != modern_manifest['identity']['embedding_hash'] or pointer['manifest']['article_ids'] != df.article_id.tolist():
            raise ValueError('Embedding timing context does not match the selected modern run')
        embedding_context = json.loads(execution_path.read_text())
        first = embedding_context['first_execution']
        for key in ['model_load_download_seconds','embedding_seconds']:
            runtime.append(dict(model='bertopic',stage=f'shared embedding preparation: {key}',seconds=first[key]))
    else:
        print('Embedding runtime context unavailable; do not treat missing costs as zero.')
    runtime_table = pd.DataFrame(runtime)
    runtime_table.to_csv(COMPARISON/'runtime_comparison.csv',index=False)
    write_json(COMPARISON/'runtime_context.json', dict(
        modern=modern_manifest.get('environment',{}), classical=classical_manifest.get('environment',{}),
        classical_manifest=str(CLASSICAL/'manifest.json'), embedding=embedding_context,
        note='Recorded stages only; missing hardware, timing, or stage definitions must be supplied before cost ranking.'))
    display(runtime_table)
    if not classical_manifest.get('timings'): print('LDA runtime pending measured stage timings.')
    stability_path = OUT/'selected'/selection['run_id']/'stability.csv'
    if stability_path.exists(): display(pd.read_csv(stability_path))
    else: print('Selected modern seed checks pending notebook 02.')
else:
    print('Runtime comparison pending both model exports.')

Runtime comparison pending both model exports.


## 6. Freeze the independent review packet
Only confirmed models/labels and the original reservation may generate a packet. Laura's manifest
must attest that the reserved articles were excluded from development review. Existing freezes
reject changed labels or assignments; reruns preserve existing answer CSVs.

Give each person [reviewer 1](../human-review/reviewer_1.ipynb) or
[reviewer 2](../human-review/reviewer_2.ipynb). Each independently reviews both models for the
same 100 articles (200 article–model items). The model identities are visible; this is not blinded.
Use the [coordinator notebook](../human-review/review_coordinator.ipynb) to monitor completion
and record qualitative interpretations. Never discuss scores before completing independent ratings.

**1–5 fit rubric:** 1 unrelated/misleading; 2 weak/incidental; 3 partial/mixed; 4 good with minor
omissions; 5 clear central-theme fit. For modern topic −1 leave fit rating blank, but complete
relevance/ambiguity flags and qualitative notes. These articles were fitted unsupervised;
independence here refers to development review and naming, not train/test separation.

In [10]:
review_folder = None
human_status = 'Pending frozen labels and complete LDA handoff.'
if COMPARISON is not None and labels_ready and classical_labels_ready:
    lookup = {('bertopic',int(row.topic_id)):row.proposed_label for row in labels.itertuples()}
    lookup.update({('lda',int(row.topic_id)):row.label for row in classical_topics.itertuples()})
    review_folder = COMPARISON/'human_review'
    packet = ev.review_packets(df, modern, classical, review_folder, lookup,
                               article_ids=reservation['article_ids'],reservation=reservation)
    human_status = 'Packet ready; independent ratings pending.'
    print('Frozen packet:', review_folder)
else:
    print(human_status)
    if classical is not None and not classical_labels_ready:
        print('Laura must confirm labels and the shared review reservation in her manifest.')

Pending frozen labels and complete LDA handoff.


## 7. Human results and combined pipeline table
After both reviewers submit, set `RUN_HUMAN_SUMMARY = True` and rerun from the top. Existing
ratings are validated against the current frozen packet before aggregation. No example or
assistant-generated scores are inserted. Incomplete reviews remain pending.

Report each reviewer's mean, exact agreement, agreement within one point, and quadratic-weighted
Cohen's kappa with a 95% interval from 1,000 paired article bootstrap resamples. Exclude modern
outliers from fit means and agreement, report their sample count/frequency separately, and discuss
their qualitative review. Undefined kappa stays missing; degenerate bootstrap resamples are counted.
LDA and modern means can have different denominators due to outliers; the table is not a paired
significance test or a ranking of accuracy. Two team reviewers provide limited subjective evidence.

In [11]:
# Enable only after both reviewers finish independently.
RUN_HUMAN_SUMMARY = False
human_summary = None
if review_folder is not None:
    # Validate files and check completion counts without displaying individual scores.
    sessions = [load_session(review_folder, reviewer) for reviewer in ['reviewer_1','reviewer_2']]
    display(pd.DataFrame([dict(reviewer=s['reviewer'],completed=int(s['complete'].sum()),total=len(s['complete'])) for s in sessions]))
    if RUN_HUMAN_SUMMARY:
        result = ev.summarize_reviews(review_folder, bootstrap_samples=1000, seed=42)
        if isinstance(result, pd.DataFrame):
            human_summary = result
            human_summary['unassigned_fraction'] = human_summary.unassigned_articles / human_summary.sample_articles
            human_status = 'complete'
            display(human_summary)
        else:
            human_status = result
    else:
        human_status = 'Pending coordinator enabling summary after both independent submissions.'
print(human_status)

pipeline_comparison = None
if comparison_metrics is not None:
    pipeline_comparison = comparison_metrics.copy()
    human_columns = ['sample_articles','rated_articles','unassigned_articles','unassigned_fraction',
                     'mean_1','mean_2','exact_agreement','within_one_agreement','weighted_kappa',
                     'kappa_ci_low','kappa_ci_high','valid_bootstrap_samples']
    if human_summary is not None:
        pipeline_comparison = pipeline_comparison.merge(human_summary[['model']+human_columns],on='model',validate='one_to_one')
    else:
        for column in human_columns: pipeline_comparison[column] = np.nan
    pipeline_comparison['human_status'] = human_status
    pipeline_comparison['fitting_representation_seconds'] = pipeline_comparison.model.map({'bertopic':modern_manifest['metrics']['fitting_representation_seconds']})
    pipeline_comparison['recorded_runtime_stages_seconds'] = pipeline_comparison.model.map({name:json.dumps(dict(zip(group.stage,group.seconds))) for name,group in runtime_table.groupby('model')})
    pipeline_comparison['runtime_detail'] = 'runtime_comparison.csv; inspect recorded stage definitions and runtime_context.json'
    pipeline_comparison.to_csv(COMPARISON/'pipeline_comparison.csv',index=False)
    display(pipeline_comparison)

Pending frozen labels and complete LDA handoff.


## 8. Interpretation and evidence for the report
Use the outputs to write article-supported conclusions; no thematic findings are inferred automatically.

- Which LDA/modern topic pairs have shared keywords **and** supporting development articles? Which matches are ambiguous or split across different granularities?
- Do both heatmaps support similar publication patterns? Name sample sizes and date ranges; use the complete matrices when a topic is absent from the main figure.
- Do human ratings support the interpretations? Discuss disagreements, ambiguous labels, irrelevant articles, and the qualitative outlier review separately. Preserve original independent scores.
- Do coherence, diversity, coverage, runtime, and human judgments point in different directions? Explain tradeoffs rather than selecting a winner using one metric.

The combined table keeps missing human values blank, and the runtime table preserves source stage
names. The checklists below distinguish produced evidence from pending team judgments. Neither
this notebook nor the coordinator edits `pr3-submission.tex`.

Held-out evaluation and shared-period publication sensitivity remain final-report work. Singleton
publication/year strata need a documented split policy; held-out experiments require training-only
preprocessing, selection, and fitting. PR2 feedback is absent; do not claim it was addressed without
the actual feedback. Do not infer sentiment, public opinion, or causality from topic shares.

In [12]:
evidence = pd.DataFrame([
    ('Methodology','Shared corpus and review reservation','complete','corpus_manifest.json; review_sample.json'),
    ('Results','Selected modern run','complete' if modern is not None else 'pending team selection','modern/selection.json'),
    ('Results','Validated LDA handoff','complete' if classical is not None else 'pending Laura','classical/{assignments,terms,topics}.csv; manifest.json'),
    ('Results','Shared metrics, overlap, publication patterns, runtime','complete' if COMPARISON is not None else 'pending both models','model_metrics.csv; jaccard.csv; assignment_contingency.csv; matched_publication_proportions.csv; runtime_comparison.csv'),
    ('Evaluation','Frozen packet','complete' if review_folder is not None else 'pending confirmed labels','human_review/review_freeze.json'),
    ('Evaluation','Independent reviewer results',human_status,'human_review/human_evaluation.csv; pipeline_comparison.csv'),
    ('Discussion','Article-supported themes and publication conclusions','pending team interpretation','topic inventories; development representatives; coordinator notes'),
    ('Limitations','Coverage, granularity, outliers, timing boundaries, two reviewers','documented; discuss observed consequences','Notebook captions and interpretation prompts')
],columns=['manuscript_section','evidence','status','artifact'])
status_folder = COMPARISON if COMPARISON is not None else COMPARISON_ROOT
# This top-level status is always current, even when no model is selected.
write_json(COMPARISON_ROOT/'status.json', dict(corpus_hash=corpus_hash(df),
    modern_run=selection['run_id'] if selection else None,
    classical_run=str(classical.run_id.iloc[0]) if classical is not None else None,
    comparison_folder=str(COMPARISON.relative_to(ROOT)) if COMPARISON is not None else None,
    human_status=human_status))
evidence.to_csv(status_folder/'comparison_evidence_checklist.csv',index=False)
display(evidence)
completion = pd.DataFrame([
    ('Select modern model and confirm both label sets','Freeze independent evaluation inputs','Team + Laura','documented choice and confirmed labels','Team to confirm'),
    ('Supply LDA handoff and measured runtime context','Enable shared quantitative comparison','Laura','validated exports and stage/hardware definitions','Team to confirm'),
    ('Complete independent ratings and qualitative outlier review','Evaluate article–topic fit','Two reviewers','original CSVs and summary with uncertainty','Team to confirm'),
    ('Interpret cross-model publication patterns','Answer how findings depend on pipeline','Team','article-supported discussion','Team to confirm'),
    ('Held-out and shared-period sensitivity','Assess generalization and coverage effects','Mary Kate + Laura + Lauren','training-only evaluation and date-restricted tables','Team to confirm')
],columns=['remaining_task','reason','responsible_member','expected_output','completion_date'])
completion.to_csv(status_folder/'completion_plan.csv',index=False)
display(completion)

,manuscript_section,evidence,status,artifact
0,Methodology,Shared corpus and review reservation,complete,corpus_manifest.json; review_sample.json
1,Results,Selected modern run,pending team selection,modern/selection.json
2,Results,Validated LDA handoff,pending Laura,"classical/{assignments,terms,topics}.csv; mani..."
3,Results,"Shared metrics, overlap, publication patterns,...",pending both models,model_metrics.csv; jaccard.csv; assignment_con...
4,Evaluation,Frozen packet,pending confirmed labels,human_review/review_freeze.json
5,Evaluation,Independent reviewer results,Pending frozen labels and complete LDA handoff.,human_review/human_evaluation.csv; pipeline_co...
6,Discussion,Article-supported themes and publication concl...,pending team interpretation,topic inventories; development representatives...
7,Limitations,"Coverage, granularity, outliers, timing bounda...",documented; discuss observed consequences,Notebook captions and interpretation prompts


,remaining_task,reason,responsible_member,expected_output,completion_date
0,Select modern model and confirm both label sets,Freeze independent evaluation inputs,Team + Laura,documented choice and confirmed labels,Team to confirm
1,Supply LDA handoff and measured runtime context,Enable shared quantitative comparison,Laura,validated exports and stage/hardware definitions,Team to confirm
2,Complete independent ratings and qualitative o...,Evaluate article–topic fit,Two reviewers,original CSVs and summary with uncertainty,Team to confirm
3,Interpret cross-model publication patterns,Answer how findings depend on pipeline,Team,article-supported discussion,Team to confirm
4,Held-out and shared-period sensitivity,Assess generalization and coverage effects,Mary Kate + Laura + Lauren,training-only evaluation and date-restricted t...,Team to confirm
